In [2]:
import numpy as np
from osgeo import gdal
from osgeo import osr
import matplotlib.pyplot as plt
from osgeo import gdal_array
import sys
sys.path.append('……/SR_ISS/')
from utils.imresize import imresize
from supres import DSen2_20

#5m-2.5m

def readtif(fname):
    # dataset = gdal.Open(DATA_PATH+fname)
    dataset = gdal.Open(fname)
    im_width = dataset.RasterXSize  
    im_height = dataset.RasterYSize  
    im_bands = dataset.RasterCount  
    im_data = dataset.ReadAsArray(0, 0, im_width, im_height)  
    return im_data

def geoinfor_tif(fname):
    # rs_data = gdal.Open(DATA_PATH+fname)
    rs_data = gdal.Open(fname)
    
    im_col = rs_data.RasterXSize
    im_row = rs_data.RasterYSize
    im_bands = rs_data.RasterCount
    im_geotrans = rs_data.GetGeoTransform()
    im_proj = rs_data.GetProjection()
    
    # espg_code = osr.SpatialReference(wkt=im_proj).GetAttrValue('AUTHORITY', 1)

    img_info = {
    'geotrans': im_geotrans,
    # 'geosrs': espg_code,
    'geosrs': im_proj,
    'row': im_row,
    'col': im_col,
    'bands': im_bands }
    return img_info

def writeTiff(im_data, im_geotrans, im_geosrs, path_out):
    '''
    input:
        im_data: tow dimentions (order: row, col),or three dimentions (order: row, col, band)
        im_geosrs: espg code correspond to image spatial reference system.
    '''
    im_data = np.squeeze(im_data)
    if 'int8' in im_data.dtype.name:
        data_type = gdal.GDT_Byte
    elif 'int16' in im_data.dtype.name:
        data_type = gdal.GDT_UInt16
    else:
        data_type = gdal.GDT_UInt32
    if len(im_data.shape) == 3:
        im_data = np.transpose(im_data)
        # im_data = im_data 
        im_bands, im_height, im_width = im_data.shape
        # im_bands, im_width, im_height = im_data.shape
        print(im_data.shape,im_bands,im_height, im_width)
    else:
        im_bands, (im_height, im_width) = 1, im_data.shape
    driver = gdal.GetDriverByName("GTiff")
    dataset = driver.Create(path_out, im_width, im_height, im_bands, data_type)
    if dataset is not None and im_geotrans is not None and im_geosrs is not None:
        dataset.SetGeoTransform(im_geotrans)
        dataset.SetProjection("EPSG:" + str(im_geosrs))
    if im_bands > 1:
        for i in range(im_bands):
            dataset.GetRasterBand(i + 1).WriteArray(im_data[i])
        del dataset
    else:
        dataset.GetRasterBand(1).WriteArray(im_data)
        del dataset

In [6]:
import os
if __name__ == '__main__':
    for i in ["1"]:
        
        dir_im_high = 'SR_ISS/data/test_data/E98N34_4328bands_2.5m_clip4.tif'
        im_high = readtif(dir_im_high)
        im_high = np.transpose(im_high)
        im_high_info = geoinfor_tif(dir_im_high)
        im_high = im_high[:,:,:4]
        dir_im_low='SR_ISS/data/test_data/E98N34_10bands_5m_clip4.tif'
        im_low = readtif(dir_im_low)
        im_low_info = geoinfor_tif(dir_im_low)
        im_low = np.transpose(im_low)
        im_low = im_low[:,:,4:]
        print(im_high.shape)
        print(im_low.shape)
        print(im_low_info)

        # SR_low = DSen2__low(im_high, im_low, deep=True)
        SR_low = DSen2__low(im_high, im_low)
        te_low = SR_low[:, :, :]
        # te_low = im_low[:, :, 4:]
        print('im_high.shape={}'.format(im_high.shape))
        print('te_low.shape={}'.format(te_low.shape))

       
        # data = np.ones((600, 600, 10), dtype=np.int16)
        x=te_low.shape[0]
        y=te_low.shape[1]
        data = np.ones((x, y, 10), dtype=np.int16)
        data[:, :, :4] = im_high
        data[:, :, 4:] = te_low

        
        out_path__high = 'SR_ISS/data/test_data/E98N34_10bands_2.5m_clip4.tif'
        writeTiff(data, im_high_info['geotrans'], im_high_info['geosrs'], out_path__high)

(1000, 1000, 4)
(500, 500, 6)
{'geotrans': (475913.6303478913, 5.0, 0.0, 3816192.2546918057, 0.0, -5.0), 'geosrs': '', 'row': 500, 'col': 500, 'bands': 10}
8 4
(1016, 1016, 4) 8 8 81
(81, 6, 128, 128)
(81, 6, 64, 64)
(81, 4, 128, 128)
[  0  56 112 168 224 280 336 392 444]
[  0  56 112 168 224 280 336 392 444]
count= 81
Symbolic Model Created.
Predicting using file: SR_ISS/data/network/s2_038_cbam_bigdatalr_1e-04.hdf5
3/3 [==============================] - 1s 183ms/step
(6, 1000, 1000)
im10.shape=(1000, 1000, 4)
te20.shape=(1000, 1000, 6)
(10, 1000, 1000) 10 1000 1000


ERROR 1: PROJ: proj_create_from_database: crs not found


In [4]:
from osgeo import gdal, osr


srs = osr.SpatialReference()
srs.ImportFromEPSG(4326)  
# out_IRS.SetProjection(srs.ExportToWkt())  


0